# Fine-Tuning — Hugging Face (Bölüm 3, Kısa Uygulama)
**Görev:** MRPC. İki cümle **aynı anlamda mı?** (`equivalent` / `not_equivalent`). Model: `bert-base-uncased`.

| Adım | Kurs Karşılığı | Ne yapılıyor? |
|---|---|---|
| 1 | Bölüm 3/2 | Veri yükleme ve tokenizasyon |
| 2 | Bölüm 3/3 | `Trainer` ile fine-tuning |
| 3 | Bölüm 3/5 | Öğrenme eğrileri |
| 4 | Uygulama | "Aynı anlamda mı?" aracı |

**Çalıştırma:** Runtime → Change runtime type → **GPU**, sonra hücreleri sırayla çalıştır (yaklaşık 3-5 dk).

## 1. Veri İşleme

In [ ]:
!pip install -q -U datasets accelerate scikit-learn matplotlib

import torch, numpy as np, matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
                          TrainingArguments, Trainer, set_seed)
from sklearn.metrics import accuracy_score, f1_score

set_seed(42)
checkpoint = "bert-base-uncased"
raw = load_dataset("nyu-mll/glue", "mrpc")
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

print(raw)
print("\nÖrnek:", raw["train"][0])   # etiket 1 = equivalent, 0 = not_equivalent

# İki cümle tokenizer'a [CLS] cümle1 [SEP] cümle2 [SEP] olarak girer
ids = tokenizer("This is the first sentence.", "This is the second one.")["input_ids"]
print("\nToken'lar:", tokenizer.convert_ids_to_tokens(ids))

tokenized = raw.map(lambda ex: tokenizer(ex["sentence1"], ex["sentence2"], truncation=True), batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)   # dinamik padding: batch içi en uzuna göre

## 2. Trainer ile Fine-Tuning
`Trainer`; optimizer (AdamW), doğrusal lr azalması, batch'leme ve GPU'ya taşımayı kendisi yapar. `compute_metrics` ile accuracy ve F1 hesaplanır, `eval_strategy="epoch"` ile her epoch sonunda doğrulama yapılır.

In [ ]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds)}

args = TrainingArguments("test-trainer", eval_strategy="epoch", logging_steps=20, num_train_epochs=3,
                         fp16=torch.cuda.is_available(), report_to="none", save_strategy="no", seed=42)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
trainer = Trainer(model=model, args=args, train_dataset=tokenized["train"], eval_dataset=tokenized["validation"],
                  data_collator=data_collator, processing_class=tokenizer, compute_metrics=compute_metrics)
trainer.train()

trainer.model.config.id2label = {0: "not_equivalent", 1: "equivalent"}
print("\nDoğrulama sonucu:", trainer.evaluate())

## 3. Öğrenme Eğrileri

In [ ]:
log = trainer.state.log_history
tr_s = [e["step"] for e in log if "loss" in e];          tr_l = [e["loss"] for e in log if "loss" in e]
ev_s = [e["step"] for e in log if "eval_loss" in e];     ev_l = [e["eval_loss"] for e in log if "eval_loss" in e]
ev_a = [e["eval_accuracy"] for e in log if "eval_loss" in e]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(tr_s, tr_l, label="Eğitim loss"); ax[0].plot(ev_s, ev_l, "o-", label="Doğrulama loss")
ax[0].set_title("Loss eğrisi"); ax[0].set_xlabel("Adım"); ax[0].legend(); ax[0].grid(True, linestyle="--", alpha=0.5)
ax[1].plot(ev_s, ev_a, "o-", color="green")
ax[1].set_title("Doğrulama accuracy"); ax[1].set_xlabel("Adım"); ax[1].grid(True, linestyle="--", alpha=0.5)
plt.tight_layout(); plt.show()
print("Doğrulama loss (epoch sonu):", [round(x, 4) for x in ev_l], "-> artıyorsa overfitting işareti")

## 4. Uygulama: "Bu İki Cümle Aynı Anlamda mı?"

In [ ]:
def paraphrase_mi(s1, s2):
    m = trainer.model.eval()
    enc = tokenizer(s1, s2, return_tensors="pt", truncation=True).to(next(m.parameters()).device)
    with torch.no_grad():
        p = m(**enc).logits.softmax(-1)[0]
    return m.config.id2label[int(p.argmax())], round(float(p.max()), 3)

names = {0: "not_equivalent", 1: "equivalent"}
print("DOĞRULAMA SETİNDEN ÖRNEKLER")
for i in range(3):
    ex = raw["validation"][i]
    label, p = paraphrase_mi(ex["sentence1"], ex["sentence2"])
    print(f"{'✔' if label == names[ex['label']] else '✘'} Gerçek: {names[ex['label']]:15s} | Model: {label} ({p})")

print("\nKENDİ ÇİFTLERİMİZ")
for a, b in [("The company announced a new product on Monday.", "On Monday, the company unveiled a new product."),
             ("The weather is lovely today.", "Stocks fell sharply on Monday."),
             ("The committee approved the budget.", "The committee rejected the budget.")]:
    print(paraphrase_mi(a, b), "|", a, "->", b)